# 🤖 HumAIne Hackathon — Active Learning + XAI for Ticket Routing

---

## Your Mission

You are building an AI system that automatically routes support tickets to the correct team — and **explains why** it made each decision.

You will:
1. Configure and partially implement an **Active Learning agent** (Cell 3)
2. Build an **XAI pipeline** using SHAP and LIME to explain routing decisions (Cell 5)

---

## Notebook Structure

| Cell | Content | Your role |
|------|---------|----------|
| **0** | **Logger setup — fill in your Student ID** | **Fill in + run first** |
| 1 | Install & imports | Run as-is |
| 2 | Load & prepare data | Run as-is |
| **3** | **AL TASK: configure + init + one cycle** | **Write code here** |
| 3b | Log AL decisions (auto) | Run as-is after Cell 3 |
| 4 | Remaining AL rounds + accuracy plot | Run as-is |
| **5** | **XAI TASK: ClassifierExplainer + predict_fn + SHAP + LIME** | **Write code here** |
| 5b | Log XAI results + finalise session (auto) | Run as-is after Cell 5 |
| 6 | Bonus: confusion matrix + SHAP waterfall | Optional |
| **7** | **Export HAIC log — mandatory** | **Run last** |

---

> ⏱️ **Both Cell 3 and Cell 5 are timed automatically.**  
> Cell 3 is done when it prints `✅ Round 1 complete`.  
> Cell 5 is done when it prints `✅ XAI cell complete`.  
> **Cell 7 is mandatory — run it even if you did not finish all tasks.**

In [ ]:
# ============================================================
# CELL 0 — LOGGER SETUP  (run this FIRST, before anything else)
# Fill in your Student ID and Group below, then run.
# ============================================================

import json, datetime, os, time as _time

# ── Fill these in before running ────────────────────────────
STUDENT_ID = "S??"     # e.g. "S03"
GROUP      = "?"       # "A" or "B"
COHORT     = "MSc"
# ────────────────────────────────────────────────────────────


class _HAICLogger:
    """Lightweight event logger for HAIC platform import.

    Captures:
      - AL configuration decisions and per-round metrics
      - XAI configuration and timing
      - Combined session summary for KPI-2.3 analysis
    """

    def __init__(self, student_id: str, group: str, cohort: str):
        self.student_id    = student_id
        self.group         = group
        self.cohort        = cohort
        self.session_start = datetime.datetime.now().isoformat()
        self.events        = []
        self.summary       = {}

    def log(self, event_type: str, **fields) -> dict:
        entry = {
            "event"    : event_type,
            "timestamp": datetime.datetime.now().isoformat(),
            **fields,
        }
        self.events.append(entry)
        return entry

    def export(self, output_dir: str = ".") -> str:
        payload = {
            "schema_version": "1.1",
            "session": {
                "student_id"   : self.student_id,
                "group"        : self.group,
                "cohort"       : self.cohort,
                "session_start": self.session_start,
                "session_end"  : datetime.datetime.now().isoformat(),
            },
            "events" : self.events,
            "summary": self.summary,
        }
        os.makedirs(output_dir, exist_ok=True)
        fname = f"haic_log_{self.student_id}_{self.group}.json"
        fpath = os.path.join(output_dir, fname)
        with open(fpath, "w") as f:
            json.dump(payload, f, indent=2)
        print(f"✅  Log exported → {fpath}")
        return fpath


assert STUDENT_ID != "S??", "⚠️  Set your STUDENT_ID before running (e.g. 'S03')!"
assert GROUP in ("A", "B"),  "⚠️  Set GROUP to 'A' or 'B' before running!"

_logger = _HAICLogger(STUDENT_ID, GROUP, COHORT)
_logger.log("session_start")

print(f"✅  Logger ready  (schema v1.1 — AL + XAI)")
print(f"   Student : {STUDENT_ID}  |  Group : {GROUP}  |  Cohort : {COHORT}")
print("\n➡️  Run Cell 1 next.")

In [ ]:
# ============================================================
# CELL 1 — PRE-FILLED: Install & imports
# Note: first run in a fresh Colab takes 2-3 minutes.
# Do not edit.
# ============================================================

import subprocess, sys, time, copy, warnings
warnings.filterwarnings('ignore')

print('Installing dependencies (this may take a moment)...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'modAL-python', 'scikit-learn', 'matplotlib',
    'explainerdashboard', 'lime', '-q'
])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio

from sklearn.datasets                import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import RandomForestClassifier
from sklearn.svm                     import SVC
from sklearn.metrics                 import accuracy_score, classification_report

from modAL.models      import ActiveLearner
from modAL.uncertainty import uncertainty_sampling, margin_sampling, entropy_sampling

from lime.lime_text     import LimeTextExplainer
from explainerdashboard import ClassifierExplainer

pio.renderers.default = 'colab'

print('\n✅ Setup complete. Run Cell 2 next.')

In [ ]:
# ============================================================
# CELL 2 — PRE-FILLED: Load & prepare data
# Run this cell. Do not edit.
# ============================================================

# 4 newsgroup categories simulate 4 support ticket routing teams.
# This is the same domain as the HumAIne ticketing pilot (HumAL).
#
#   0 → Hardware Team
#   1 → Software Team
#   2 → Design Team
#   3 → Electronics Team

CATEGORIES = [
    'comp.sys.ibm.pc.hardware',
    'comp.os.ms-windows.misc',
    'comp.graphics',
    'sci.electronics'
]
TEAM_NAMES = ['Hardware', 'Software', 'Design', 'Electronics']

print('Loading ticket data...')
train_raw = fetch_20newsgroups(
    subset='train', categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'), random_state=42
)
test_raw = fetch_20newsgroups(
    subset='test', categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'), random_state=42
)

# ── Raw texts (needed by LIME in Cell 5) ─────────────────────
train_texts = pd.Series(train_raw.data, name='text').fillna('')
test_texts  = pd.Series(test_raw.data,  name='text').fillna('')

# ── TF-IDF vectorisation ──────────────────────────────────────
vectorizer    = TfidfVectorizer(max_features=3000, stop_words='english')
X_train_full  = vectorizer.fit_transform(train_raw.data).toarray()
X_test_np     = vectorizer.transform(test_raw.data).toarray()
feature_names = vectorizer.get_feature_names_out()

y_train_full  = train_raw.target
y_test_np     = test_raw.target

# ── DataFrames for ClassifierExplainer (Cell 5) ───────────────
X_test_df     = pd.DataFrame(X_test_np,  columns=feature_names)
y_test_series = pd.Series(y_test_np, name='target')

# ── Cold-start seed: 12 labeled tickets (3 per team) ─────────
N_INITIAL   = 12
np.random.seed(42)
initial_idx = np.concatenate([
    np.random.choice(np.where(y_train_full == t)[0], size=3, replace=False)
    for t in range(4)
])

X_initial = X_train_full[initial_idx]
y_initial = y_train_full[initial_idx]
X_pool    = np.delete(X_train_full, initial_idx, axis=0)
y_pool    = np.delete(y_train_full, initial_idx, axis=0)

# X_test stays as numpy for the AL loop; X_test_df is for XAI
X_test = X_test_np
y_test = y_test_np

print('\n📋 Data ready!')
print(f'   Labeled seed  : {len(X_initial)} tickets  ({N_INITIAL//4} per team)')
print(f'   Unlabeled pool: {len(X_pool)} tickets')
print(f'   Test set      : {len(X_test)} tickets')
print(f'   Teams         : {", ".join(TEAM_NAMES)}')
print(f'   Vocabulary    : {len(feature_names)} TF-IDF features')
print('\n✅ Ready. Fill in and run Cell 3 — AL TASK.')

---
## ✏️ CELL 3 — AL TASK: Configure + Initialize + One Cycle

**Part A** — Make three configuration decisions (replace each `???`)  
**Part B.1** — Initialize the `ActiveLearner` from the modAL library  
**Part B.2** — Write one complete query-teach cycle (4 steps)

**Use whatever resources are available to you.**

---

In [ ]:
# ============================================================
# CELL 3 — AL TASK
# ⏱️  AL timer starts now.
# Goal: reach the final print without errors.
# ============================================================

_t_al_start = time.time()   # ← do not remove


# ────────────────────────────────────────────────────────────
# PART A — Three decisions (replace each ???)
# ────────────────────────────────────────────────────────────

# DECISION 1 — Base classifier
# Options:
#   LogisticRegression(max_iter=1000)
#   RandomForestClassifier(n_estimators=100)
#   SVC(kernel='rbf', probability=True)
#
# Think about: which handles high-dimensional sparse TF-IDF text
# well AND supports probability estimates (required for uncertainty
# query strategies and for LIME in Cell 5)?

MY_CLASSIFIER = ???


# DECISION 2 — Query strategy
# Options:
#   uncertainty_sampling  → most uncertain single prediction
#   margin_sampling       → smallest gap between top-2 classes
#   entropy_sampling      → highest uncertainty across ALL classes
#
# Think about: with 4 teams and only 12 initial labels,
# which strategy best handles multi-class cold-start?

MY_QUERY_STRATEGY = ???


# DECISION 3 — Queries per round
# Pick a number between 5 and 30.
#
# Think about: 12 initial labels, 10 rounds, 4 teams.
# How many total labeled tickets will you accumulate?

N_QUERIES_PER_ROUND = ???


# ────────────────────────────────────────────────────────────
# PART B.1 — Initialize the ActiveLearner
#
# Create an object called `learner` using modAL's ActiveLearner.
#
# Required keyword arguments:
#   estimator       — classifier from Decision 1
#   query_strategy  — strategy from Decision 2
#   X_training      — initial labeled features  (X_initial)
#   y_training      — initial labeled targets   (y_initial)
#
# The ActiveLearner fits the classifier automatically on init.
# ────────────────────────────────────────────────────────────

# YOUR CODE ↓



# ────────────────────────────────────────────────────────────
# PART B.2 — One complete query-teach cycle
#
#   Step 1 — QUERY
#     learner.query(X_pool, n_instances=N_QUERIES_PER_ROUND)
#     Returns: (query_idx, query_instance)
#
#   Step 2 — RETRIEVE LABELS
#     new_labels = y_pool[query_idx]
#
#   Step 3 — TEACH
#     learner.teach(X_pool[query_idx], new_labels)
#
#   Step 4 — REMOVE FROM POOL
#     np.delete(array, query_idx, axis=0)  ← do for BOTH X_pool and y_pool
# ────────────────────────────────────────────────────────────

# Step 1 — Query
# YOUR CODE ↓


# Step 2 — Retrieve labels
# YOUR CODE ↓


# Step 3 — Teach
# YOUR CODE ↓


# Step 4 — Remove from pool (both X_pool and y_pool)
# YOUR CODE ↓



# ────────────────────────────────────────────────────────────
# Verification — do not edit below this line
# ────────────────────────────────────────────────────────────

_t_al_end   = time.time()
_al_time    = _t_al_end - _t_al_start

assert 'learner' in dir(), 'learner not defined — did you complete Part B.1?'
assert isinstance(learner, ActiveLearner), 'learner must be an ActiveLearner instance.'
assert len(X_pool) == len(y_pool), 'X_pool and y_pool differ — check Step 4.'
assert len(X_pool) < len(X_train_full) - N_INITIAL, 'Pool unchanged — did you run Step 4?'

_acc_r1         = accuracy_score(y_test, learner.predict(X_test))
_labeled_so_far = N_INITIAL + N_QUERIES_PER_ROUND

print('✅ Round 1 complete!')
print(f'   Classifier        : {type(MY_CLASSIFIER).__name__}')
print(f'   Query strategy    : {MY_QUERY_STRATEGY.__name__}')
print(f'   Queries per round : {N_QUERIES_PER_ROUND}')
print(f'   Labeled so far    : {_labeled_so_far}')
print(f'   Pool remaining    : {len(X_pool)}')
print(f'   Accuracy after R1 : {_acc_r1:.3f}')
print(f'\n⏱️  AL configuration time : {_al_time:.1f} s')
print('\n📋 RECORD → AL config time =', round(_al_time, 1), 's')
print('\n➡️  Run Cell 3b (logging), then Cell 4.')

In [ ]:
# ============================================================
# CELL 3b — AUTO: Log AL decisions + Round 1 metrics
# Run immediately after Cell 3 succeeds. Do not edit.
# ============================================================

# Log the three configuration decisions
_logger.log(
    'al_configuration',
    classifier          = type(MY_CLASSIFIER).__name__,
    query_strategy      = MY_QUERY_STRATEGY.__name__,
    n_queries_per_round = int(N_QUERIES_PER_ROUND),
    al_config_time_s    = float(round(_al_time, 2)),
)

# Log Round 1 outcome
_logger.log(
    'round_complete',
    round          = 1,
    labeled_so_far = int(N_INITIAL + N_QUERIES_PER_ROUND),
    pool_remaining = int(len(X_pool)),
    accuracy       = float(round(_acc_r1, 4)),
    config_time_s  = float(round(_al_time, 2)),
)

print(f'📝  AL config logged  |  config_time={_al_time:.1f}s  |  acc_r1={_acc_r1:.3f}')
print('\n➡️  Run Cell 4.')

In [ ]:
# ============================================================
# CELL 4 — PRE-FILLED: Remaining AL rounds + accuracy plot
# Picks up from Round 1 in Cell 3. Do not edit.
# ============================================================

_t_loop = time.time()

# Baseline: accuracy with only 12 labels, no AL
_base_clf = copy.deepcopy(MY_CLASSIFIER)
_base_clf.fit(X_initial, y_initial)
baseline_acc = accuracy_score(y_test, _base_clf.predict(X_test))

accuracy_history = [baseline_acc, _acc_r1]
labeled_counts   = [N_INITIAL,    N_INITIAL + N_QUERIES_PER_ROUND]

_Xp = X_pool.copy()
_yp = y_pool.copy()

print(f'Running 9 more rounds...\n')
print(f'{"Round":>5}  {"Labeled":>8}  {"Accuracy":>9}  {"Change":>8}')
print('-' * 40)
print(f'{"Base":>5}  {N_INITIAL:>8}  {baseline_acc:>9.3f}  {"—":>8}')
print(f'{1:>5}  {labeled_counts[1]:>8}  {_acc_r1:>9.3f}  {_acc_r1-baseline_acc:>+8.3f}  ← you wrote this')

for i in range(9):
    n_q = min(N_QUERIES_PER_ROUND, len(_Xp))
    if n_q == 0:
        print('Pool exhausted early.')
        break
    qi, _ = learner.query(_Xp, n_instances=n_q)
    learner.teach(_Xp[qi], _yp[qi])
    _Xp = np.delete(_Xp, qi, axis=0)
    _yp = np.delete(_yp, qi, axis=0)
    acc = accuracy_score(y_test, learner.predict(X_test))
    accuracy_history.append(acc)
    labeled_counts.append(labeled_counts[-1] + n_q)
    print(f'{i+2:>5}  {labeled_counts[-1]:>8}  {acc:>9.3f}  {acc-accuracy_history[-2]:>+8.3f}')

    # ── HAIC log: one event per round ─────────────────────────
    _logger.log(
        'round_complete',
        round          = i + 2,
        labeled_so_far = int(labeled_counts[-1]),
        pool_remaining = int(len(_Xp)),
        accuracy       = float(round(acc, 4)),
        delta_accuracy = float(round(acc - accuracy_history[-2], 4)),
    )
    # ──────────────────────────────────────────────────────────

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(labeled_counts, accuracy_history, marker='o', color='steelblue',
        linewidth=2, label='AL accuracy')
ax.axhline(baseline_acc, color='gray', linestyle='--', alpha=0.7,
           label=f'Baseline ({N_INITIAL} labels)')
ax.scatter([labeled_counts[1]], [_acc_r1], color='orange', s=120,
           zorder=5, label='Round 1 (you wrote this)')
ax.set_xlabel('Labeled tickets'); ax.set_ylabel('Test accuracy')
ax.set_title('Active Learning: Ticket Routing Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

_final_al_acc = accuracy_history[-1]
_al_loop_time = time.time() - _t_loop

print(f'\n✅ AL complete  |  Baseline: {baseline_acc:.3f}  →  Final: {_final_al_acc:.3f}')
print(f'   Total labels used : {labeled_counts[-1]}')
print('\n➡️  Now run Cell 5 — XAI TASK.')

---
## ✏️ CELL 5 — XAI TASK: Explain the Model's Routing Decisions

Your Active Learner can route tickets — but **why** does it send a ticket to 'Electronics' instead of 'Hardware'?  
This cell builds the XAI pipeline that answers that question.

**Part A** — Wrap your trained learner in a `ClassifierExplainer` (HumAIne's XAI wrapper)  
**Part B** — Write a `predict_fn` closure so LIME can perturb raw ticket text  
**Part C** — Plot global SHAP feature importance  
**Part D** — Run LIME on one ticket

**Use whatever resources are available to you.**

---

In [ ]:
# ============================================================
# CELL 5 — XAI TASK
# ⏱️  XAI timer starts now.
# Goal: reach the final print without errors.
# ============================================================

_t_xai_start = time.time()   # ← do not remove


# ────────────────────────────────────────────────────────────
# PART A — Wrap the trained learner in a ClassifierExplainer
#
# Create an object called `explainer` using ClassifierExplainer.
#
# Positional arguments (in order):
#   1. learner.estimator
#      → pass the underlying sklearn model, NOT learner itself.
#        ClassifierExplainer inspects the model type to pick the
#        right SHAP backend (LinearExplainer for LogisticRegression).
#
#   2. X_test_df
#      → pandas DataFrame with named TF-IDF columns (already defined).
#
#   3. y_test_series
#      → pandas Series of true labels (already defined).
#
# Keyword argument:
#   labels=TEAM_NAMES  → list of class names for readable plot labels.
# ────────────────────────────────────────────────────────────

# YOUR CODE ↓



# ────────────────────────────────────────────────────────────
# PART B — Write the predict_fn closure for LIME
#
# LIME perturbs RAW TEXT and calls your predict_fn with many
# perturbed strings. The function must reproduce the exact
# TF-IDF preprocessing: raw text → features → probabilities.
#
# Write a function called `predict_fn` that:
#   1. Accepts `texts` — a list of raw strings
#   2. Vectorizes:  X_sparse = vectorizer.transform(pd.Series(texts).fillna(''))
#   3. Converts:    X_df = pd.DataFrame(X_sparse.toarray(), columns=feature_names)
#   4. Returns:     learner.predict_proba(X_df)
#
# Common mistakes: forgetting .fillna(''), forgetting .toarray(),
# using predict() instead of predict_proba().
# ────────────────────────────────────────────────────────────

# YOUR CODE ↓  (def predict_fn(texts): ...)



# ────────────────────────────────────────────────────────────
# PART C — SHAP: global feature importance
#
# Step 1: fig = explainer.plot_importances()
# Step 2: fig.show()
# ────────────────────────────────────────────────────────────

# YOUR CODE ↓  (2 lines)



# ────────────────────────────────────────────────────────────
# PART D — LIME: explain one individual ticket
#
# Step 1: lime_exp = LimeTextExplainer(class_names=TEAM_NAMES)
# Step 2: exp = lime_exp.explain_instance(
#             test_texts.iloc[0], predict_fn,
#             num_features=10, top_labels=2)
# Step 3: exp.show_in_notebook(text=True)
# ────────────────────────────────────────────────────────────

# YOUR CODE ↓  (3 lines)



# ────────────────────────────────────────────────────────────
# Verification — do not edit below this line
# ────────────────────────────────────────────────────────────

_t_xai_end  = time.time()
_xai_time   = _t_xai_end - _t_xai_start

assert 'explainer' in dir(), 'explainer not defined — did you complete Part A?'
assert isinstance(explainer, ClassifierExplainer), 'explainer must be a ClassifierExplainer.'
assert callable(predict_fn), 'predict_fn must be a callable function.'

_test_proba = predict_fn([test_texts.iloc[0]])
assert _test_proba.shape == (1, 4), (
    f'predict_fn must return shape (1, 4), got {_test_proba.shape}. '
    f'Check vectorizer.transform() and .toarray().'
)
assert abs(_test_proba.sum() - 1.0) < 0.01, (
    'predict_fn probabilities do not sum to 1. Are you calling predict_proba?'
)

_shap_backend   = type(explainer.shap_explainer).__name__
_predicted_team = TEAM_NAMES[int(_test_proba.argmax())]

print('\n✅ XAI cell complete!')
print(f'   SHAP backend       : {_shap_backend}')
print(f'   predict_fn shape   : {_test_proba.shape}  |  sum = {_test_proba.sum():.3f}')
print(f'   Ticket #0 routed → : {_predicted_team} team')
print(f'\n⏱️  XAI configuration time : {_xai_time:.1f} s')
print('\n📋 RECORD → XAI config time =', round(_xai_time, 1), 's')
print('📋 RECORD → Total config time =', round(_al_time + _xai_time, 1), 's')
print('\n➡️  Run Cell 5b (logging), then Cell 6 if you have time, then Cell 7 (mandatory).')

In [ ]:
# ============================================================
# CELL 5b — AUTO: Log XAI results + finalise session summary
# Run immediately after Cell 5 succeeds. Do not edit.
# ============================================================

# Log XAI configuration and timing
_logger.log(
    'xai_configuration',
    shap_backend          = _shap_backend,
    xai_config_time_s     = float(round(_xai_time, 2)),
    predict_fn_output_shape = list(_test_proba.shape),
    predicted_team_ticket0  = _predicted_team,
)

# Build the full session summary (AL + XAI combined)
_total_time = _al_time + _al_loop_time + _xai_time

_logger.summary = {
    # ── AL fields (mirrors schema v1.0) ───────────────────────
    'classifier'          : type(MY_CLASSIFIER).__name__,
    'query_strategy'      : MY_QUERY_STRATEGY.__name__,
    'n_queries_per_round' : int(N_QUERIES_PER_ROUND),
    'al_config_time_s'    : float(round(_al_time, 2)),
    'n_rounds'            : len(accuracy_history) - 1,
    'total_labels_used'   : int(labeled_counts[-1]),
    'baseline_accuracy'   : float(round(baseline_acc, 4)),
    'final_al_accuracy'   : float(round(_final_al_acc, 4)),
    'al_improvement'      : float(round(_final_al_acc - baseline_acc, 4)),
    'accuracy_history'    : [float(round(a, 4)) for a in accuracy_history],
    'labeled_counts'      : labeled_counts,
    # ── XAI fields (new in schema v1.1) ───────────────────────
    'shap_backend'        : _shap_backend,
    'xai_config_time_s'   : float(round(_xai_time, 2)),
    # ── Combined KPI-2.3 field ────────────────────────────────
    'total_config_time_s' : float(round(_al_time + _xai_time, 2)),
    'total_session_time_s': float(round(_total_time, 2)),
}

_logger.log(
    'session_complete',
    final_al_accuracy   = float(round(_final_al_acc, 4)),
    xai_complete        = True,
    total_config_time_s = float(round(_al_time + _xai_time, 2)),
)

print('📝  XAI results logged.')
print(f'   AL config time  : {_al_time:.1f} s')
print(f'   XAI config time : {_xai_time:.1f} s')
print(f'   Total config    : {_al_time + _xai_time:.1f} s')
print(f'   Final AL acc    : {_final_al_acc:.3f}')
print('\n➡️  Run Cell 6 (bonus) or go straight to Cell 7 (mandatory export).')

---
## 🌟 BONUS — Only if you have time

Confusion matrix + SHAP waterfall for one ticket.

---

In [ ]:
# ============================================================
# CELL 6 — BONUS: Confusion matrix + SHAP waterfall
# Pre-filled. Run only after Cell 5b.
# ============================================================

from sklearn.metrics import confusion_matrix

# Confusion matrix
y_pred = learner.predict(X_test)
print(classification_report(y_test, y_pred, target_names=TEAM_NAMES))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(TEAM_NAMES, rotation=30, ha='right')
ax.set_yticklabels(TEAM_NAMES)
ax.set_xlabel('Predicted team'); ax.set_ylabel('Actual team')
ax.set_title('Confusion Matrix — Ticket Routing')
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

# SHAP waterfall for ticket #0
print(f'\nSHAP waterfall for ticket #0:')
print(f'  Preview: "{test_texts.iloc[0][:120]}..."')
print(f'  Predicted: {TEAM_NAMES[int(learner.predict(X_test[:1])[0])]}')
fig2 = explainer.plot_shap_contributions(idx=0)
fig2.show()

print('\n💬 Discussion questions:')
print('   1. Which two teams are confused most? Why?')
print('   2. Do SHAP (global) and LIME (local) highlight the same words?')
print('   3. How would you change your query strategy based on this confusion matrix?')

In [ ]:
# ============================================================
# CELL 7 — EXPORT HAIC LOG  ← MANDATORY — run this last
#
# Saves your complete session log as a JSON file.
# Upload this file to the shared lab folder when done.
# Run this cell even if you did not finish all tasks.
# ============================================================

_log_path = _logger.export(output_dir='.')

print()
print('📤  Please upload this file to the shared lab folder:')
print(f'    {_log_path}')
print()
print('The log contains:')
print(f'   {len(_logger.events)} events logged')
print(f'   Student: {STUDENT_ID}  |  Group: {GROUP}')
if _logger.summary:
    print(f'   AL config time  : {_logger.summary.get("al_config_time_s", "not recorded")} s')
    print(f'   XAI config time : {_logger.summary.get("xai_config_time_s", "not recorded")} s')
    print(f'   Total config    : {_logger.summary.get("total_config_time_s", "not recorded")} s')
    print(f'   Final AL acc    : {_logger.summary.get("final_al_accuracy", "not recorded")}')
else:
    print('   (partial log — session did not complete)')